In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# R03 — Anti-collapse objectives and gradient paths

Read the single primary lesson before this lab. Predict each result. Numerical correctness is not independent mastery. See `representations/SOURCE_AUDIT.md` for provenance.

In [2]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / 'src/shape_lab').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook within the extracted course project.')
sys.path.insert(0, str(ROOT / 'src'))
from shape_lab import representations as r
from shape_lab import representation_experiments as exp
print('Course source:', ROOT)


Course source: /tmp/shape-course-q90rdcpz/course


## The exact same views have zero agreement error, even when constant.

In [3]:
a=np.zeros((5,3)); b=np.zeros((5,3))
assert np.mean((a-b)**2)==0
print('Zero agreement error is not evidence of informative features.')

Zero agreement error is not evidence of informative features.


## Inspect objective controls built from actual training images.

In [4]:
result=exp.mechanisms_lab()
print(json.dumps(result,indent=2))
assert result['info_nce_same'] < result['info_nce_shuffled']
assert result['zero_cloud_penalty'] > 0
assert result['zero_cloud_gradient_norm'] == 0
assert result['stop_gradient_target_is_none']

{
  "data": "64 real training digit images, fixed random projections and manufactured perturbations",
  "agreement_same": 0.00038117991323442026,
  "agreement_different": 30.097293919693143,
  "vicreg_same": 33.25053640984614,
  "barlow_same": 0.01992710355773961,
  "info_nce_same": 3.433419644504835,
  "info_nce_shuffled": 5.4280279930539175,
  "zero_cloud_penalty": 13.085463046861499,
  "zero_cloud_gradient_norm": 0.0,
  "stop_gradient_online": -2.0,
  "stop_gradient_target_is_none": true,
  "ema_hand_example": 1.5,
  "interpretation": "Separate objective scales are not a model ranking. A positive anti-collapse penalty does not ensure a nonzero gradient at every state."
}


## Check the Torch gradient against finite differences, rather than trusting autodiff usage alone.

In [5]:
import torch
x=np.random.default_rng(5).normal(size=(8,3)); A=r.unit_directions(3,8,6)
z=torch.tensor(x,dtype=torch.float64,requires_grad=True)
loss=r.sigreg_torch(z,torch.tensor(A)); loss.backward()
h=1e-5; plus=x.copy(); minus=x.copy(); plus[0,0]+=h; minus[0,0]-=h
fd=(r.projected_gaussian_score(plus,A)-r.projected_gaussian_score(minus,A))/(2*h)
print({'autodiff':z.grad[0,0].item(),'finite_difference':fd})
assert np.isclose(z.grad[0,0].item(),fd,rtol=1e-5)

{'autodiff': -0.16494917038757723, 'finite_difference': -0.16494917038922807}


## Your evidence
Record: observation unit; assumptions; prediction; calculated result; difference from your prediction; and one unsupported conclusion. Attempt the separate learner notebook before opening the reference solutions.